In [4]:
import sys

print("Python executable:", sys.executable)
print("Python version:", sys.version)

# install yfinance if not already installed
!{sys.executable} -m pip install yfinance

Python executable: /usr/local/bin/python3
Python version: 3.12.0 (v3.12.0:0fb18b02c8, Oct  2 2023, 09:45:56) [Clang 13.0.0 (clang-1300.0.29.30)]

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [6]:
import yfinance as yf
import pandas as pd
from datetime import datetime

ticker = "SPY"
tk = yf.Ticker(ticker)

# 1. Underlying price history (for realized vol / sanity checks)
hist = tk.history(period="2y")
hist.to_csv(f"data/{ticker}_history.csv")

# 2. Risk-free rate proxy (13-week T-bill)
irx = yf.Ticker("^IRX").history(period="5d")["Close"].iloc[-1] / 100
print(f"risk-free rate proxy: {irx:.4f}")

# 3. Available expiries
expiries = tk.options
print(expiries)

# 4. Pull chains for a handful of expiries (mix of short- and longer-dated)
chains = []
for exp in expiries[:6]:
    chain = tk.option_chain(exp)
    calls = chain.calls.assign(expiry=exp, type="call")
    puts = chain.puts.assign(expiry=exp, type="put")
    chains.append(pd.concat([calls, puts]))

option_data = pd.concat(chains, ignore_index=True)
option_data["mid"] = (option_data["bid"] + option_data["ask"]) / 2
option_data["pull_date"] = datetime.today().strftime("%Y-%m-%d")
option_data.to_csv(f"data/{ticker}_options_raw.csv", index=False)

risk-free rate proxy: 0.0367
('2026-07-06', '2026-07-07', '2026-07-08', '2026-07-09', '2026-07-10', '2026-07-17', '2026-07-24', '2026-07-31', '2026-08-07', '2026-08-21', '2026-08-31', '2026-09-18', '2026-09-30', '2026-10-16', '2026-10-30', '2026-11-20', '2026-11-30', '2026-12-18', '2026-12-31', '2027-01-15', '2027-03-19', '2027-03-31', '2027-06-17', '2027-09-17', '2027-12-17', '2028-01-21', '2028-06-16', '2028-12-15')
